# Moduł 06. Python Unit Tests — `unittest.mock` Podstawy — v2.0

Autor: Kamil Bartocha

## Rozkład jazdy

- 🔹 1. Rodzaje atrap - Mock, Stub, Spy, Fake
- 🔹 2. `Mock` i `MagicMock` - tworzenie i konfiguracja
- 🔹 3. `patch` - podmiana obiektów w testach

## 1. 🔹 Rodzaje atrap - Mock, Stub, Spy, Fake

Testowanie izolowane wymaga zastępowania zewnętrznych zależności
atrapami (test doubles). Rozróżniamy cztery podstawowe rodzaje:

- **Stub** - zwraca predefiniowane wartości, nie weryfikuje wywołań
- **Mock** - weryfikuje sposób użycia (czy i jak był wywoływany)
- **Spy** - opakowanie prawdziwego obiektu, rejestruje wywołania
- **Fake** - uproszczona, działająca implementacja (np. baza in-memory)

W Pythonie moduł `unittest.mock` dostarcza klasy `Mock` i `MagicMock`,
które pokrywają role stubu i mocka jednocześnie. Importujemy je przez:

```python
from unittest.mock import Mock, MagicMock, patch
```

Atrapa (mock) nie wykonuje żadnej prawdziwej logiki - odpowiada
skonfigurowanymi wartościami i zapamiętuje każde wywołanie.

In [ ]:
from unittest.mock import Mock, MagicMock, patch

# Stub - zwraca stałą wartość
db_stub = Mock()
db_stub.get_user.return_value = {'id': 1, 'name': 'Alice'}
print('stub:', db_stub.get_user(1))

# Mock - weryfikuje wywołanie
service_mock = Mock()
service_mock.send_email('alice@example.com', 'Hello')
service_mock.send_email.assert_called_once_with('alice@example.com', 'Hello')
print('mock called:', service_mock.send_email.called)

# Fake - prosta implementacja zastępcza
class FakeRepository:
    def __init__(self):
        self._data = {}

    def save(self, key, value):
        self._data[key] = value

    def get(self, key):
        return self._data.get(key)

fake_repo = FakeRepository()
fake_repo.save('user:1', {'name': 'Bob'})
print('fake:', fake_repo.get('user:1'))

---

### 🐍 Ćwiczenia — rodzaje atrap

1. Utwórz `Mock()` i sprawdź `return_value`, `called`, `call_count`
2. Skonfiguruj `side_effect` jako wyjątek i jako listę wartości

In [ ]:
from unittest.mock import Mock

# Ćwiczenie 1: Mock - return_value, called, call_count
# Utwórz mock dla metody calculate(a, b) zwracającej 42.
# Wywołaj go trzy razy i sprawdź: called, call_count.

calc_mock = Mock()
calc_mock.calculate.return_value = ...

# wywołaj trzy razy
...

print('called:', calc_mock.calculate.called)       # True
print('call_count:', calc_mock.calculate.call_count)  # 3
print('return_value:', calc_mock.calculate(5, 7))  # 42

In [ ]:
from unittest.mock import Mock
import unittest

# Ćwiczenie 2: side_effect jako wyjątek i lista wartości
# a) Skonfiguruj mock.fetch tak, by rzucał ValueError przy wywołaniu.
# b) Skonfiguruj mock.roll tak, by kolejno zwracał [3, 1, 4, 1, 5].

# a) side_effect jako wyjątek
fetch_mock = Mock()
fetch_mock.fetch.side_effect = ...

try:
    fetch_mock.fetch('/bad-url')
except ValueError as e:
    print('wyjątek:', e)

# b) side_effect jako lista
roll_mock = Mock()
roll_mock.roll.side_effect = ...

results = [roll_mock.roll() for _ in range(5)]
print('rzuty:', results)  # [3, 1, 4, 1, 5]

## 2. 🔹 `Mock` i `MagicMock` - tworzenie i konfiguracja

Klasa `Mock` tworzy dynamiczny obiekt akceptujący dowolne atrybuty
i wywołania. `MagicMock` rozszerza `Mock` o obsługę magicznych
metod (`__len__`, `__iter__`, `__enter__`, `__exit__` i innych).

Kluczowe atrybuty i metody:

| Atrybut / metoda | Opis |
|---|---|
| `return_value` | wartość zwracana przy wywołaniu |
| `side_effect` | wyjątek lub iterowalny wynik |
| `called` | `True` jeśli wywołany przynajmniej raz |
| `call_count` | liczba wywołań |
| `call_args` | argumenty ostatniego wywołania |
| `assert_called_once_with(*a, **kw)` | weryfikacja jednego wywołania |
| `assert_called_with(*a, **kw)` | weryfikacja ostatniego wywołania |
| `reset_mock()` | reset liczników |

Parametr `spec=` ogranicza mock do interfejsu wskazanego obiektu -
wywołanie nieistniejącej metody podnosi `AttributeError`.

In [ ]:
from unittest.mock import Mock, MagicMock

# Mock z return_value
m = Mock()
m.connect.return_value = True
print('connect:', m.connect('localhost', 5432))

# side_effect jako lista
m.read.side_effect = [b'chunk1', b'chunk2', StopIteration]
print('read1:', m.read())  # b'chunk1'
print('read2:', m.read())  # b'chunk2'

# assert_called_with
m.connect('db.local', 3306)
m.connect.assert_called_with('db.local', 3306)
print('assert_called_with: OK')

# MagicMock - magiczne metody
mm = MagicMock()
mm.__len__.return_value = 7
mm.__iter__.return_value = iter([10, 20, 30])
print('len:', len(mm))           # 7
print('iter:', list(mm))         # [10, 20, 30]

# spec=
class RealService:
    def process(self, data): pass

strict_mock = Mock(spec=RealService)
strict_mock.process({'key': 'val'})
try:
    strict_mock.nonexistent()
except AttributeError as e:
    print('spec blokuje:', e)

---

### 🐍 Ćwiczenia — Mock i MagicMock

3. *(Trudniejsze)* `MagicMock` dla klasy z `__len__` i `__iter__`
7. Sprawdź `assert_called_once_with`, `assert_called_with`
8. Użyj `call_args_list` do weryfikacji wielu wywołań
10. Mock dla metody klasy - `patch.object`
11. Użyj `spec=` by mock respektował interfejs oryginału

In [ ]:
from unittest.mock import MagicMock

# Ćwiczenie 3: MagicMock z __len__ i __iter__ *(Trudniejsze)*
# Stwórz MagicMock symulujący kolekcję 5 elementów: [1, 2, 3, 4, 5].
# Skonfiguruj __len__ i __iter__, a następnie sprawdź:
#   len(mock_collection) == 5
#   list(mock_collection) == [1, 2, 3, 4, 5]
# hint: __iter__ musi zwracać nowy iterator przy każdym wywołaniu.

mock_collection = MagicMock()
mock_collection.__len__.return_value = ...
mock_collection.__iter__.return_value = ...

print('len:', len(mock_collection))         # 5
print('list:', list(mock_collection))       # [1, 2, 3, 4, 5]

In [ ]:
from unittest.mock import Mock
import unittest

# Ćwiczenie 7: assert_called_once_with, assert_called_with
# Utwórz mock dla serwisu powiadomień.
# Wywołaj notify('user1', 'Witaj!'), a następnie zweryfikuj:
#   - assert_called_once_with('user1', 'Witaj!')
# Wywołaj ponownie z innymi arg i użyj assert_called_with.

notifier = Mock()

notifier.notify(...)
...

notifier.notify('user2', 'Nowa wiadomość')
...

print('call_count:', notifier.notify.call_count)  # 2

In [ ]:
from unittest.mock import Mock, call

# Ćwiczenie 8: call_args_list
# Utwórz mock logger.log i wywołaj go 3 razy:
#   log('INFO', 'start'), log('WARNING', 'slow'), log('ERROR', 'fail')
# Wypisz call_args_list i sprawdź, że drugi wpis to call('WARNING','slow').

logger = Mock()

...

print('call_args_list:', logger.log.call_args_list)
assert logger.log.call_args_list[1] == call(...)
print('OK')

In [ ]:
from unittest.mock import patch, Mock
import unittest

# Ćwiczenie 10: patch.object
# Klasa Payment.charge(amount) wywołuje self._gateway.process(amount).
# Użyj patch.object, by podmienić metodę _gateway.process i sprawdzić
# że charge(100) wywołuje process(100) i zwraca True.

class Gateway:
    def process(self, amount):
        raise RuntimeError('prawdziwy gateway - nie wywoływać!')

class Payment:
    def __init__(self):
        self._gateway = Gateway()

    def charge(self, amount):
        return self._gateway.process(amount)

payment = Payment()

with patch.object(payment._gateway, 'process', return_value=True) as mock_proc:
    result = payment.charge(...)
    mock_proc.assert_called_once_with(...)
    print('charge result:', result)   # True

In [ ]:
from unittest.mock import Mock

# Ćwiczenie 11: spec=
# Zdefiniuj klasę UserRepository z metodami find(id) i save(user).
# Utwórz Mock(spec=UserRepository) i sprawdź:
#   a) wywołanie find(1) działa poprawnie
#   b) wywołanie delete(1) podnosi AttributeError

class UserRepository:
    def find(self, user_id): ...
    def save(self, user): ...

repo_mock = Mock(spec=...)
repo_mock.find.return_value = {'id': 1, 'name': 'Alice'}

print('find:', repo_mock.find(1))

try:
    repo_mock.delete(1)
    print('Błąd: powinien rzucić AttributeError')
except AttributeError:
    print('spec blokuje nieistniejącą metodę: OK')

## 3. 🔹 `patch` - podmiana obiektów w testach

Dekorator / context manager `patch` zastępuje wskazany obiekt
w przestrzeni nazw testowanego modułu atrapą na czas testu.
Po zakończeniu testu oryginał jest przywracany automatycznie.

Trzy formy użycia:

```python
# 1. Dekorator
@patch('mymodule.requests.get')
def test_api(mock_get):
    mock_get.return_value.json.return_value = {'key': 'val'}
    ...

# 2. Context manager
with patch('mymodule.requests.get') as mock_get:
    ...

# 3. patch.object - podmiana metody na istniejącym obiekcie
with patch.object(obj, 'method', return_value=42):
    ...
```

Ścieżka do patchowania wskazuje **gdzie jest używany** obiekt,
nie skąd pochodzi. Np. jeśli `mymodule.py` robi `import os`,
patchujemy `mymodule.os.path.exists`, nie `os.path.exists`.

`mock_open` to pomocnicza funkcja do mockowania wbudowanej `open`:

```python
from unittest.mock import patch, mock_open
with patch('builtins.open', mock_open(read_data='hello')):
    content = open('any.txt').read()  # 'hello'
```

In [ ]:
import sys, subprocess, tempfile, os

def _run(code, flags=None):
    flags = flags or ['-v', '--tb=short', '-p', 'no:cacheprovider']
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='_test.py', delete=False, dir='.'
    ) as tmp:
        tmp.write(code)
        fname = tmp.name
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', fname] + flags,
        capture_output=True, text=True
    )
    short = os.path.basename(fname)
    print(result.stdout.replace(fname, short)[-1500:])
    os.unlink(fname)

# Przykład 1: patch jako dekorator
_run('''
from unittest.mock import patch, mock_open
import json

def fetch_config(path):
    with open(path) as f:
        return json.load(f)

@patch('builtins.open', mock_open(read_data=json.dumps({"debug": True})))
def test_fetch_config_reads_file():
    config = fetch_config("/etc/app.json")
    assert config["debug"] is True
''')

# Przykład 2: patch jako context manager
_run('''
from unittest.mock import patch
import os

def file_exists_message(path):
    if os.path.exists(path):
        return f"{path} exists"
    return f"{path} missing"

def test_file_exists():
    with patch("os.path.exists", return_value=True) as mock_exists:
        msg = file_exists_message("/some/file.txt")
        assert msg == "/some/file.txt exists"
        mock_exists.assert_called_once_with("/some/file.txt")

def test_file_missing():
    with patch("os.path.exists", return_value=False):
        msg = file_exists_message("/no/file.txt")
        assert msg == "/no/file.txt missing"
''')

---

### 🐍 Ćwiczenia — patch

4. Użyj `@patch('module.ClassName')` do podmiany w dekoratorze
5. Użyj `patch()` jako context manager
6. *(Trudniejsze)* Patch `requests.get` w teście funkcji pobierającej
   dane z API
9. *(Trudniejsze)* Patch wbudowanej funkcji `open` przez `mock_open`
12. *(Trudniejsze)* Przetestuj klasę `EmailSender` mockując SMTP

In [ ]:
from unittest.mock import patch, Mock
import unittest

# Ćwiczenie 4: @patch jako dekorator
# Funkcja send_report(user_id) korzysta z database.Database().get_user(id).
# Użyj @patch('__main__.Database'), by podmienić klasę Database.
# Sprawdź, że send_report zwraca oczekiwany komunikat.

class Database:
    def get_user(self, user_id):
        raise RuntimeError('prawdziwa baza - nie wywoływać!')

def send_report(user_id):
    db = Database()
    user = db.get_user(user_id)
    return f"Report for {user['name']}"


class TestSendReport(unittest.TestCase):
    @patch('__main__.Database')
    def test_send_report(self, MockDB):
        MockDB.return_value.get_user.return_value = ...
        result = send_report(1)
        self.assertEqual(result, ...)
        print('result:', result)

unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
from unittest.mock import patch, Mock
import unittest

# Ćwiczenie 5: patch() jako context manager
# Funkcja get_timestamp() wywołuje datetime.datetime.now().
# Użyj patch() jako context manager, by zwrócić stałą datę
# i sprawdzić, że get_timestamp() zwraca oczekiwany string.
# hint: patchuj 'datetime.datetime'

import datetime

def get_timestamp():
    return datetime.datetime.now().strftime('%Y-%m-%d')


class TestTimestamp(unittest.TestCase):
    def test_get_timestamp(self):
        fixed = datetime.datetime(2024, 1, 15)
        with patch('datetime.datetime') as mock_dt:
            mock_dt.now.return_value = ...
            result = get_timestamp()
            self.assertEqual(result, ...)
            print('timestamp:', result)

unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
from unittest.mock import patch, Mock
import unittest

# Ćwiczenie 6: Patch requests.get *(Trudniejsze)*
# Funkcja fetch_weather(city) wywołuje requests.get(url) i zwraca
# temperature z odpowiedzi JSON.
# Podmień requests.get przez @patch i sprawdź zwracaną wartość.
# hint: mock_response.json.return_value = {'temperature': 22}

import requests

def fetch_weather(city):
    url = f'https://api.weather.example.com/{city}'
    response = requests.get(url)
    data = response.json()
    return data['temperature']


class TestFetchWeather(unittest.TestCase):
    @patch('requests.get')
    def test_fetch_weather(self, mock_get):
        mock_response = Mock()
        mock_response.json.return_value = ...
        mock_get.return_value = mock_response

        temp = fetch_weather('Warsaw')
        self.assertEqual(temp, ...)
        mock_get.assert_called_once_with(...)
        print('temperature:', temp)

unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
from unittest.mock import patch, mock_open
import unittest

# Ćwiczenie 9: mock_open *(Trudniejsze)*
# Funkcja read_config(path) otwiera plik i parsuje linie jako
# klucz=wartość słownik. Podmień open() przez mock_open,
# by symulować zawartość pliku bez prawdziwego pliku.
# hint: mock_open(read_data='host=localhost\nport=5432\n')

def read_config(path):
    config = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if '=' in line:
                key, val = line.split('=', 1)
                config[key] = val
    return config


class TestReadConfig(unittest.TestCase):
    def test_read_config(self):
        fake_content = ...
        with patch('builtins.open', mock_open(read_data=fake_content)):
            config = read_config('/etc/app.conf')
        self.assertEqual(config['host'], 'localhost')
        self.assertEqual(config['port'], '5432')
        print('config:', config)

unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
from unittest.mock import patch, Mock, MagicMock
import unittest

# Ćwiczenie 12: EmailSender z SMTP *(Trudniejsze)*
# Klasa EmailSender.send(to, subject, body) używa smtplib.SMTP.
# Podmień smtplib.SMTP (jako context manager) i sprawdź:
#   - SMTP został wywołany z ('smtp.example.com', 587)
#   - sendmail wywołany z właściwymi argumentami
# hint: SMTP zwrócony przez __enter__ ma metodę sendmail

import smtplib

class EmailSender:
    def __init__(self, host, port):
        self.host = host
        self.port = port

    def send(self, to, subject, body):
        with smtplib.SMTP(self.host, self.port) as smtp:
            smtp.sendmail('from@example.com', to,
                          f'Subject: {subject}\n\n{body}')


class TestEmailSender(unittest.TestCase):
    @patch('smtplib.SMTP')
    def test_send_email(self, MockSMTP):
        mock_smtp = MagicMock()
        MockSMTP.return_value.__enter__.return_value = mock_smtp

        sender = EmailSender('smtp.example.com', 587)
        sender.send('alice@example.com', 'Test', 'Hello!')

        MockSMTP.assert_called_once_with(...)
        mock_smtp.sendmail.assert_called_once()
        args = mock_smtp.sendmail.call_args[0]
        self.assertEqual(args[1], ...)
        print('sendmail args:', args)

unittest.main(argv=[''], exit=False, verbosity=2)